# POSet Shock Validation Framework

Validate whether countries occupying **higher POSet levels** exhibit stronger resilience during systemic shocks.

### Assumptions
- The POSet construction has already been completed.

### Input files
| File | Required columns |
|---|---|
| `poset_results.csv` | `Country`, `profile_code` |
| `shock_validation_data.csv` | `Country`, `year`, `GDP_Growth`, `Inflation`, `Unemployment`, `Productivity` |

### Outputs (per shock + combined)
- `country_resilience_scores_{shock_id}.csv`
- `level_validation_results_{shock_id}.csv`
- `statistical_validation_{shock_id}.csv`
- `combined_resilience_scores.csv`
- `combined_level_validation.csv`
- `combined_statistical_validation.csv`


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kruskal, spearmanr
from sklearn.preprocessing import RobustScaler


## Configuration

Each shock is defined by a pre-shock baseline window and a shock window.


In [3]:
SHOCKS = [
    {
        "id":          "GFC_2008",
        "label":       "Global Financial Crisis (2008)",
        "pre_start":   2005,
        "pre_end":     2007,
        "shock_start": 2008,
        "shock_end":   2010,
    },
    {
        "id":          "COVID_2020",
        "label":       "COVID-19 Pandemic (2020)",
        "pre_start":   2017,
        "pre_end":     2019,
        "shock_start": 2020,
        "shock_end":   2022,
    },
]

MISSING_THRESHOLD = 0.10

VALIDATION_COLUMNS = [
    "GDP_Growth",
    "Inflation_Rate",
    "Unemployment_Rate",
    "Productivity_Variation",
]

METRICS = [
    "GDP_Resilience",
    "Unemployment_Resilience",
    "Productivity_Resilience",
    "Inflation_Stability",
    "Recovery_Speed",
]



## Load Data

In [4]:
print("cwd:", os.getcwd())
print("exists:", Path("data/Processed/dataset_macro_performance.csv").exists())

poset_path = (
    Path("data") / "Processed" / "Profile_Epsilon_Margin_POSet"
    / "profile_epsilon_margin_poset_2019_5levels_eps_0_5_country_profiles.csv"
)
shock_path = Path("data") / "Processed" / "dataset_macro_performance.csv"

poset_df = pd.read_csv(poset_path)
shock_df  = pd.read_csv(shock_path)

print(f"POSet  : {poset_df.shape[0]} rows, {poset_df.shape[1]} cols")
print(f"Shock  : {shock_df.shape[0]} rows, {shock_df.shape[1]} cols")
print(poset_df.head())
print(shock_df.head())


cwd: c:\Documenti\UNIMIB\Data Science Lab\PROJECT
exists: True
POSet  : 37 rows, 13 cols
Shock  : 888 rows, 8 cols
   Unnamed: 0 country  Energy_Import_Dependency_scaled  \
0          18     AUS                         1.000000   
1          42     AUT                         0.255237   
2          66     BEL                         0.090698   
3          90     CAN                         1.000000   
4         114     CHE                         0.445968   

   Public_Debt_pct_GDP_scaled  R&D_Expenditure_pct_GDP_scaled  \
0                    0.707326                        0.268728   
1                    0.644087                        0.516220   
2                    0.490815                        0.517572   
3                    0.745337                        0.262496   
4                    0.942857                        0.505528   

   Tertiary_Education_Attainment_scaled  Gov_Score_scaled  \
0                              0.708329          0.840545   
1                      

## Preprocessing

### Remove High-Missing Countries


In [5]:
def remove_high_missing_countries(df, threshold):
    missing_ratio = (
        df.groupby("country")
        .apply(lambda x: x.isna().mean().mean())
    )
    valid_countries = missing_ratio[missing_ratio <= threshold].index
    return df[df["country"].isin(valid_countries)]

shock_df = remove_high_missing_countries(shock_df, MISSING_THRESHOLD)
print(f"Countries after missing filter: {shock_df['country'].nunique()}")


Countries after missing filter: 37


C:\Users\3003f\AppData\Local\Temp\ipykernel_4916\3087335739.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isna().mean().mean())


### Country-wise Interpolation

In [6]:
shock_df = shock_df.sort_values(["country", "year"])

for col in VALIDATION_COLUMNS:
    # Linear interpolation within each country
    shock_df[col] = (
        shock_df.groupby("country")[col]
        .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
    )
    # Remaining NaNs → country mean
    shock_df[col] = (
        shock_df.groupby("country")[col]
        .transform(lambda x: x.fillna(x.mean()))
    )

print("Interpolation complete.")
print(shock_df[VALIDATION_COLUMNS].isna().sum().rename("Remaining NaN"))


Interpolation complete.
GDP_Growth                0
Inflation_Rate            0
Unemployment_Rate         0
Productivity_Variation    0
Name: Remaining NaN, dtype: int64


## Shock Resilience Metrics

For each country and shock episode we compute:

| Metric | Direction | Definition |
|---|---|---|
| `GDP_Resilience` | ↑ higher = better | worst shock GDP − baseline GDP |
| `Unemployment_Resilience` | ↑ higher = better | baseline unemp − worst shock unemp |
| `Productivity_Resilience` | ↑ higher = better | worst shock prod − baseline prod |
| `Inflation_Stability` | ↑ higher = better | negative std of shock-period inflation |
| `Recovery_Speed` | ↑ higher = better | negative years to recover baseline GDP |


In [16]:
def compute_shock_resilience(df, shock_cfg):
    """Compute per-country resilience metrics for a single shock episode."""
    pre_start, pre_end     = shock_cfg["pre_start"],   shock_cfg["pre_end"]
    shock_start, shock_end = shock_cfg["shock_start"], shock_cfg["shock_end"]

    results = []

    for country in df["country"].unique():
        cdf = df[df["country"] == country]

        pre_shock    = cdf[(cdf["year"] >= pre_start)   & (cdf["year"] <= pre_end)]
        shock_period = cdf[(cdf["year"] >= shock_start) & (cdf["year"] <= shock_end)]

        if len(pre_shock) == 0 or len(shock_period) == 0:
            continue

        # GDP resilience (higher = better)
        baseline_gdp   = pre_shock["GDP_Growth"].mean()
        worst_gdp      = shock_period["GDP_Growth"].min()
        gdp_resilience = worst_gdp - baseline_gdp

        # Unemployment resilience (lower increase = better)
        baseline_unemp          = pre_shock["Unemployment_Rate"].mean()
        worst_unemp             = shock_period["Unemployment_Rate"].max()
        unemployment_resilience = baseline_unemp - worst_unemp

        # Productivity resilience
        baseline_prod           = pre_shock["Productivity_Variation"].mean()
        worst_prod              = shock_period["Productivity_Variation"].min()
        productivity_resilience = worst_prod - baseline_prod

        # Inflation stability (lower volatility = better)
        inflation_stability = -(shock_period["Inflation_Rate"].std())

        # Recovery speed (faster return to baseline GDP = better)
        recovery_year = None
        for _, row in shock_period.iterrows():
            if row["GDP_Growth"] >= baseline_gdp:
                recovery_year = row["year"]
                break

        recovery_score = (
            -(recovery_year - shock_start)
            if recovery_year is not None
            else -(shock_end - shock_start + 1)
        )

        results.append({
            "country":                 country,
            "Shock_ID":                shock_cfg["id"],
            "GDP_Resilience":          gdp_resilience,
            "Unemployment_Resilience": unemployment_resilience,
            "Productivity_Resilience": productivity_resilience,
            "Inflation_Stability":     inflation_stability,
            "Recovery_Speed":          recovery_score,
        })

    return pd.DataFrame(results)


## Normalisation & Composite Score

`RobustScaler` is applied to each raw metric; the composite `Shock_Resilience_Score`
is the row-wise mean of the five scaled metrics.


In [17]:
def normalise_and_score(resilience_df):
    """RobustScaler + composite Shock_Resilience_Score."""
    scaler = RobustScaler()
    resilience_df[METRICS] = scaler.fit_transform(resilience_df[METRICS])
    resilience_df["Shock_Resilience_Score"] = resilience_df[METRICS].mean(axis=1)
    return resilience_df


## Validation Helpers

In [18]:
def level_summary(merged_df):
    """Descriptive statistics per POSet level."""
    out = (
        merged_df
        .groupby("profile_code")["Shock_Resilience_Score"]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )
    out.columns = ["profile_code", "Mean_Resilience", "Median_Resilience",
                   "Std_Resilience", "N_Countries"]
    return out


def statistical_tests(merged_df):
    """Spearman correlation + Kruskal-Wallis across POSet levels."""
    spearman_corr, spearman_p = spearmanr(
        merged_df["profile_code"], merged_df["Shock_Resilience_Score"]
    )
    groups = [
        merged_df[merged_df["profile_code"] == lvl]["Shock_Resilience_Score"]
        for lvl in sorted(merged_df["profile_code"].unique())
    ]
    kruskal_stat, kruskal_p = kruskal(*groups)

    return pd.DataFrame({
        "Metric": ["Spearman_Correlation", "Spearman_PValue",
                   "Kruskal_Statistic",    "Kruskal_PValue"],
        "Value":  [spearman_corr, spearman_p, kruskal_stat, kruskal_p],
    })


## Main Loop — One Run per Shock Episode

In [22]:
all_resilience = []
all_level      = []
all_stats      = []

output_dir = Path("Data/Processed/Shock_Validation")
output_dir.mkdir(parents=True, exist_ok=True)

for shock_cfg in SHOCKS:
    shock_id = shock_cfg["id"]

    print("=" * 60)
    print(f"PROCESSING SHOCK: {shock_cfg['label']}")
    print("=" * 60)

    # Compute + normalise
    res_df = compute_shock_resilience(shock_df, shock_cfg)
    res_df = normalise_and_score(res_df)

    # Merge with POSet levels
    merged = pd.merge(
        poset_df[["country", "profile_code"]],
        res_df,
        on="country",
        how="inner",
    )

    # Level summary + statistical tests
    lv = level_summary(merged)
    lv["Shock_ID"] = shock_id

    st = statistical_tests(merged)
    st["Shock_ID"] = shock_id

    # Print results
    print()
    print("LEVEL-BASED RESILIENCE")
    print(lv.drop(columns="Shock_ID").to_string(index=False))
    print()

    def val(metric):
        return st.loc[st["Metric"] == metric, "Value"].values[0]

    print(f"Spearman Correlation : {val('Spearman_Correlation'):.4f}")
    print(f"Spearman P-Value     : {val('Spearman_PValue'):.4f}")
    print(f"Kruskal Statistic    : {val('Kruskal_Statistic'):.4f}")
    print(f"Kruskal P-Value      : {val('Kruskal_PValue'):.4f}")
    print()

    if val("Spearman_Correlation") < 0:
        print("Validation: higher POSet levels → lower resilience  ✓")
    else:
        print("No negative monotonic relationship detected.")
    print()

    # Per-shock CSVs
    res_df.to_csv(output_dir / f"country_resilience_scores_{shock_id}.csv",  index=False)
    lv.to_csv(    output_dir / f"level_validation_results_{shock_id}.csv",   index=False)
    st.to_csv(    output_dir / f"statistical_validation_{shock_id}.csv",     index=False)

    # Accumulate
    merged["Shock_ID"] = shock_id
    all_resilience.append(merged)
    all_level.append(lv)
    all_stats.append(st)


PROCESSING SHOCK: Global Financial Crisis (2008)

LEVEL-BASED RESILIENCE
 profile_code  Mean_Resilience  Median_Resilience  Std_Resilience  N_Countries
          101         0.387292           0.387292             NaN            1
          110        -0.354592          -0.354592             NaN            1
          112         0.123999           0.123999             NaN            1
          122        -0.697848          -0.697848             NaN            1
          423         0.123065           0.123065             NaN            1
         2441         0.073330           0.073330             NaN            1
         4031        -1.837748          -1.837748             NaN            1
         4144         0.521587           0.521587             NaN            1
        11200        -0.003213          -0.003213             NaN            1
        11413         0.266723           0.266723             NaN            1
        11440         0.249152           0.249152         

## Combined Output

In [23]:
combined_resilience = pd.concat(all_resilience, ignore_index=True)
combined_level      = pd.concat(all_level,      ignore_index=True)
combined_stats      = pd.concat(all_stats,      ignore_index=True)

# Average resilience score per country across both shocks
avg_resilience = (
    combined_resilience
    .groupby("country")
    .agg(
        profile_code=("profile_code", "first"),
        Mean_Shock_Resilience_Score=("Shock_Resilience_Score", "mean"),
        N_Shocks=("Shock_Resilience_Score", "count"),
    )
    .reset_index()
)

# Average level statistics across both shocks
avg_level = (
    combined_level
    .groupby("profile_code")
    .agg(
        Mean_Resilience=("Mean_Resilience",   "mean"),
        Median_Resilience=("Median_Resilience", "mean"),
        Std_Resilience=("Std_Resilience",    "mean"),
        N_Countries=("N_Countries",      "mean"),
    )
    .reset_index()
)


# Save
avg_resilience.to_csv(output_dir / "combined_resilience_scores.csv",     index=False)
avg_level.to_csv(output_dir / "combined_level_validation.csv",      index=False)
combined_stats.to_csv(output_dir / "combined_statistical_validation.csv", index=False)

print("Combined CSVs saved.")


Combined CSVs saved.


## Combined Validation Summary

In [24]:
print("=" * 60)
print("COMBINED VALIDATION SUMMARY (both shocks)")
print("=" * 60)
print()
print(avg_level.to_string(index=False))
print()

for shock_cfg in SHOCKS:
    shock_id = shock_cfg["id"]
    sub = combined_stats[combined_stats["Shock_ID"] == shock_id]

    def val(metric):
        return sub.loc[sub["Metric"] == metric, "Value"].values[0]

    print(f"--- {shock_cfg['label']} ---")
    print(f"  Spearman r = {val('Spearman_Correlation'):.4f}  "
          f"(p = {val('Spearman_PValue'):.4f})")
    print(f"  Kruskal H  = {val('Kruskal_Statistic'):.4f}  "
          f"(p = {val('Kruskal_PValue'):.4f})")
    print()

print("=" * 60)


COMBINED VALIDATION SUMMARY (both shocks)

 profile_code  Mean_Resilience  Median_Resilience  Std_Resilience  N_Countries
          101         0.293038           0.293038             NaN          1.0
          110        -0.170375          -0.170375             NaN          1.0
          112        -0.021300          -0.021300             NaN          1.0
          122        -0.565740          -0.565740             NaN          1.0
          423         0.219679           0.219679             NaN          1.0
         2441         0.179637           0.179637             NaN          1.0
         4031        -1.255998          -1.255998             NaN          1.0
         4144         0.499071           0.499071             NaN          1.0
        11200        -0.180662          -0.180662             NaN          1.0
        11413         0.133476           0.133476             NaN          1.0
        11440         0.194820           0.194820             NaN          1.0
        1